# IMDB Sentiment Analysis: Data Preparation and Preprocessing

## 1. Introduction

Sentiment analysis identifies the attitude expressed in a piece of text.
In this project, we study binary sentiment classification of English
movie reviews using the `stanfordnlp/imdb` dataset from Hugging Face.

Each labeled review belongs to one of two classes:
- **0 — Negative**
- **1 — Positive**

The broader project will compare TF-IDF-based classifiers with a
transformer-based approach. This notebook prepares reproducible data
for those later experiments.

### Objectives of this notebook

1. Download and inspect the IMDB dataset.
2. Preserve the official test split and divide the official training
   split into fixed training and validation sets.
3. Describe dataset sizes, label distributions, and sample reviews.
4. Implement and explain tokenization, stop-word removal, and lemmatization.
5. Export data for subsequent model training.

We will preserve the original reviews and produce separate text versions
for classical models and transformer models.

**Scope:** This notebook covers data preparation and preprocessing.
Model training and performance evaluation will be conducted separately.

## 2. Environment Setup

We use the following Python libraries:

- **datasets:** Downloads and manages the Hugging Face dataset.
- **pandas:** Creates tables for inspection and reporting.
- **matplotlib:** Visualizes label distributions and review lengths.
- **spaCy:** Provides English tokenization, stop-word information,
  and lemmatization.

The spaCy English model supplies linguistic information used for
lemmatization. It is a preprocessing resource, not our sentiment classifier.

We set the random seed to **42** for the training/validation split.
We will also record package versions and save split membership to support
reproducibility.

In [ ]:
%pip install -q datasets pandas matplotlib spacy
!python -m spacy download en_core_web_sm

In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import random

import pandas as pd
import matplotlib.pyplot as plt
import spacy
from datasets import load_dataset

SEED = 42
DATASET_NAME = "stanfordnlp/imdb"
VALIDATION_FRACTION = 0.20

random.seed(SEED)

# Relative paths work in Colab, Kaggle, and local notebooks.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

package_versions = {
    package: version(package)
    for package in [
        "datasets",
        "pandas",
        "matplotlib",
        "spacy",
        "en-core-web-sm",
    ]
}

print(f"Dataset: {DATASET_NAME}")
print(f"Random seed: {SEED}")
print(f"Validation fraction of original train: {VALIDATION_FRACTION:.0%}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

pd.DataFrame(
    package_versions.items(),
    columns=["Package", "Version"],
)

In [ ]:
# Keep the components needed for context-aware lemmatization.
# Named-entity recognition and dependency parsing are unnecessary here.
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"],
)

example = "The actors were running, but the movie was not exciting."
doc = nlp(example)

pd.DataFrame({
    "Token": [token.text for token in doc],
    "Lemma": [token.lemma_ for token in doc],
    "Is stop word": [token.is_stop for token in doc],
})

## 3. Download and Inspect the Dataset

We use `stanfordnlp/imdb`, available through Hugging Face Datasets.
It contains English movie reviews for binary sentiment classification.

The dataset provides three original splits:

| Split | Number of reviews | Sentiment labels |
|---|---:|---|
| train | 25,000 | Negative or positive |
| test | 25,000 | Negative or positive |
| unsupervised | 50,000 | Unlabeled |

The labeled data uses:
- **0:** Negative sentiment
- **1:** Positive sentiment

This project uses only the labeled training and test splits.
The unsupervised split is excluded because our current task is
supervised sentiment classification.

A validation split will be created from the original training split
in a later section. The official test split will remain reserved for
final evaluation.

Source: https://huggingface.co/datasets/stanfordnlp/imdb

In [ ]:
# Load only the two splits needed for supervised classification.
dataset = load_dataset(
    DATASET_NAME,
    split={
        "train": "train",
        "test": "test",
    },
)

print(dataset)

In [ ]:
print("Dataset features:")
print(dataset["train"].features)

label_feature = dataset["train"].features["label"]

label_mapping = {
    label_id: label_feature.int2str(label_id)
    for label_id in range(label_feature.num_classes)
}

print("\nLabel mapping:")
print(label_mapping)

split_summary = pd.DataFrame([
    {
        "Original split": split_name,
        "Reviews": len(split_data),
        "Columns": ", ".join(split_data.column_names),
    }
    for split_name, split_data in dataset.items()
])

display(split_summary)

In [ ]:
# Select examples from training data.
# These illustrate the data; they are not a random sample.
examples = {}

for row in dataset["train"]:
    label_id = row["label"]

    if label_id not in examples:
        examples[label_id] = row["text"]

    if len(examples) == label_feature.num_classes:
        break

for label_id in sorted(examples):
    print(f"Label {label_id}: {label_mapping[label_id]}")
    print("Review excerpt:")
    print(examples[label_id][:700])
    print()

### Initial Inspection

The loaded data contains 25,000 training reviews and 25,000 test reviews,
giving 50,000 labeled reviews in total.

Each record has two fields:
- `text`: The original movie review.
- `label`: The negative or positive sentiment category.

The examples above show excerpts from one training review per class.
They illustrate the dataset format rather than representing all reviews.

We have not modified the review text or created the validation split yet.
The next step is to check data quality before preparing fixed splits.

## 4. Data Quality Checks

Before splitting or preprocessing, we check for:

- Missing or non-string review text.
- Empty or whitespace-only reviews.
- Unexpected sentiment labels.
- Repeated review text within each original split.
- Identical reviews associated with conflicting labels.
- Identical review text appearing in both training and test data.

Repeated text can affect evaluation if the same review appears in
multiple splits. Conflicting labels may indicate ambiguous annotations
or data-quality issues.

These checks inspect the original text without modifying or deleting
reviews. Duplicate counts refer to exact text matches; near-duplicate
reviews may not be detected.

In [ ]:
# Keep the Hugging Face dataset unchanged.
# Create separate tables for inspection and later processing.
original_frames = {}

for split_name, split_data in dataset.items():
    frame = split_data.to_pandas()

    # The source has no review-ID column.
    # Identify each record by its original split and row position.
    frame.insert(
        0,
        "review_id",
        [f"{split_name}_{i:05d}" for i in range(len(frame))],
    )

    frame.insert(1, "original_split", split_name)
    original_frames[split_name] = frame

display(original_frames["train"].head(3))

In [ ]:
quality_rows = []

for split_name, frame in original_frames.items():
    text = frame["text"]
    labels = frame["label"]

    is_string = text.map(lambda value: isinstance(value, str))
    valid_text = text[is_string]

    empty_text = valid_text.str.strip().eq("")

    # Count extra occurrences after the first copy.
    duplicate_extras = valid_text.duplicated().sum()

    # Count distinct review texts appearing more than once.
    text_counts = valid_text.value_counts()
    repeated_texts = text_counts.gt(1).sum()

    # Find identical texts associated with more than one label.
    label_counts_per_text = (
        frame.loc[is_string]
        .groupby("text")["label"]
        .nunique()
    )
    conflicting_texts = label_counts_per_text.gt(1).sum()

    quality_rows.append({
        "Split": split_name,
        "Rows": len(frame),
        "Missing text": int(text.isna().sum()),
        "Non-string text (excluding missing)": int(
            ((~is_string) & text.notna()).sum()
        ),
        "Empty text": int(empty_text.sum()),
        "Missing labels": int(labels.isna().sum()),
        "Unexpected non-missing labels": int(
            (labels.notna() & ~labels.isin([0, 1])).sum()
        ),
        "Duplicate extra rows": int(duplicate_extras),
        "Distinct repeated texts": int(repeated_texts),
        "Texts with conflicting labels": int(conflicting_texts),
    })

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary.T)

In [ ]:
all_original = pd.concat(
    original_frames.values(),
    ignore_index=True,
)

valid_original = all_original[
    all_original["text"].map(lambda value: isinstance(value, str))
]

train_texts = set(
    original_frames["train"]["text"].dropna()
)
test_texts = set(
    original_frames["test"]["text"].dropna()
)

shared_texts = train_texts & test_texts

shared_rows = valid_original[
    valid_original["text"].isin(shared_texts)
]

conflicting_shared_texts = (
    shared_rows.groupby("text")["label"]
    .nunique()
    .gt(1)
    .sum()
)

overlap_summary = pd.DataFrame([{
    "Distinct texts shared by train and test": len(shared_texts),
    "Train rows containing shared text": int(
        original_frames["train"]["text"].isin(shared_texts).sum()
    ),
    "Test rows containing shared text": int(
        original_frames["test"]["text"].isin(shared_texts).sum()
    ),
    "Shared texts with conflicting labels": int(
        conflicting_shared_texts
    ),
}])

display(overlap_summary.T)

### Interpretation of Data Quality Checks

The tables above report data-quality findings for the original
labeled training and test splits.

No records have been removed or modified at this stage.

Missing text, empty text, and unexpected labels must be investigated
before further processing. Repeated reviews and overlap between splits
will inform our splitting policy and the limitations discussed in
the report.

Exact-text matching does not detect every near-duplicate review.

## 5. Prepare Fixed Training, Validation, and Test Sets

The quality checks found exact duplicate reviews within the original
splits and 123 distinct review texts shared by training and test data.

We apply the following policy:

1. Preserve the official test set unchanged.
2. Remove original training reviews whose exact text appears in test.
3. Keep only the first occurrence of each remaining training review text.
4. Split the remaining training data into 80% training and 20% validation,
   stratified by sentiment label, using random seed 42.

Stratification maintains approximately the same sentiment proportions
in training and validation.

The test set retains its internal duplicates for consistency with the
official benchmark. This limitation will be documented in the report.

Overlap removal uses text identity only, not test labels. Test data will
not be used to choose preprocessing rules or model settings.

These checks prevent exact-text overlap. They do not guarantee the
absence of near-duplicate reviews.

In [ ]:
original_train = original_frames["train"]
original_test = original_frames["test"]

# Step 1: remove training reviews whose exact text occurs in test.
overlap_mask = original_train["text"].isin(original_test["text"])

train_without_overlap = original_train.loc[
    ~overlap_mask
].copy()

# Step 2: keep the first occurrence of each remaining review text.
duplicate_mask = train_without_overlap["text"].duplicated(
    keep="first"
)

eligible_train = train_without_overlap.loc[
    ~duplicate_mask
].copy().reset_index(drop=True)

# Preserve every original test row and its order.
fixed_test = original_test.copy().reset_index(drop=True)

removal_summary = pd.DataFrame([
    {
        "Stage": "Original training data",
        "Rows removed": 0,
        "Rows remaining": len(original_train),
    },
    {
        "Stage": "Remove exact overlap with test",
        "Rows removed": int(overlap_mask.sum()),
        "Rows remaining": len(train_without_overlap),
    },
    {
        "Stage": "Remove remaining duplicate extra rows",
        "Rows removed": int(duplicate_mask.sum()),
        "Rows remaining": len(eligible_train),
    },
])

display(removal_summary)

In [ ]:
from datasets import Dataset

# Retain review IDs and original split information.
eligible_dataset = Dataset.from_pandas(
    eligible_train,
    preserve_index=False,
)

# Hugging Face stratification requires a ClassLabel column.
eligible_dataset = eligible_dataset.cast_column(
    "label",
    label_feature,
)

train_val = eligible_dataset.train_test_split(
    test_size=VALIDATION_FRACTION,
    seed=SEED,
    stratify_by_column="label",
)

# The "test" key here is our validation portion, not the official test.
prepared_frames = {
    "train": train_val["train"].to_pandas(),
    "val": train_val["test"].to_pandas(),
    "test": fixed_test.copy(),
}

for split_name, frame in prepared_frames.items():
    frame["split"] = split_name

print("Fixed project splits created.")

In [ ]:
split_rows = []

for split_name, frame in prepared_frames.items():
    counts = frame["label"].value_counts()
    total = len(frame)

    negative = int(counts.get(0, 0))
    positive = int(counts.get(1, 0))

    split_rows.append({
        "Split": split_name,
        "Reviews": total,
        "Negative": negative,
        "Positive": positive,
        "Negative (%)": round(100 * negative / total, 2),
        "Positive (%)": round(100 * positive / total, 2),
    })

fixed_split_summary = pd.DataFrame(split_rows)

display(fixed_split_summary)
print(
    "Total retained labeled reviews:",
    sum(len(frame) for frame in prepared_frames.values()),
)

In [ ]:
from itertools import combinations

# Check basic validity and unique record IDs within each split.
for split_name, frame in prepared_frames.items():
    assert frame["review_id"].is_unique, (
        f"Repeated review IDs in {split_name}"
    )
    assert frame["text"].map(
        lambda text: isinstance(text, str) and bool(text.strip())
    ).all(), f"Invalid review text in {split_name}"

    assert frame["label"].isin([0, 1]).all(), (
        f"Invalid labels in {split_name}"
    )

# Check record IDs and exact text across every pair of splits.
overlap_checks = []

for left, right in combinations(prepared_frames, 2):
    left_frame = prepared_frames[left]
    right_frame = prepared_frames[right]

    shared_ids = (
        set(left_frame["review_id"])
        & set(right_frame["review_id"])
    )
    shared_reviews = (
        set(left_frame["text"])
        & set(right_frame["text"])
    )

    assert not shared_ids, f"Record overlap: {left} / {right}"
    assert not shared_reviews, f"Exact-text overlap: {left} / {right}"

    overlap_checks.append({
        "Split pair": f"{left} / {right}",
        "Shared review IDs": len(shared_ids),
        "Shared exact texts": len(shared_reviews),
    })

# Every eligible training record must belong to exactly one new split.
assert (
    set(prepared_frames["train"]["review_id"])
    | set(prepared_frames["val"]["review_id"])
) == set(eligible_train["review_id"])

# Training and validation must each have unique review text.
assert prepared_frames["train"]["text"].is_unique
assert prepared_frames["val"]["text"].is_unique

# Confirm the official test records, text, labels, and order are unchanged.
pd.testing.assert_frame_equal(
    prepared_frames["test"][original_test.columns].reset_index(drop=True),
    original_test.reset_index(drop=True),
)

display(pd.DataFrame(overlap_checks))
print("All split checks passed.")

In [ ]:
split_manifest = pd.concat(
    [
        frame[["review_id", "original_split", "split", "label"]]
        for frame in prepared_frames.values()
    ],
    ignore_index=True,
)

removed_overlap = original_train.loc[
    overlap_mask, ["review_id", "original_split", "label"]
].copy()
removed_overlap["removal_reason"] = "exact_text_overlap_with_test"

removed_duplicates = train_without_overlap.loc[
    duplicate_mask, ["review_id", "original_split", "label"]
].copy()
removed_duplicates["removal_reason"] = "duplicate_training_text"

removed_records = pd.concat(
    [removed_overlap, removed_duplicates],
    ignore_index=True,
)

split_manifest.to_csv(
    OUTPUT_DIR / "split_manifest.csv",
    index=False,
)
removed_records.to_csv(
    OUTPUT_DIR / "removed_training_records.csv",
    index=False,
)
fixed_split_summary.to_csv(
    OUTPUT_DIR / "split_summary.csv",
    index=False,
)

print("Saved:")
print(OUTPUT_DIR / "split_manifest.csv")
print(OUTPUT_DIR / "removed_training_records.csv")
print(OUTPUT_DIR / "split_summary.csv")

## 6. Dataset Description

After removing overlapping and duplicate training records, the project
contains 49,781 labeled reviews:

- **Training:** 19,824 reviews
- **Validation:** 4,957 reviews
- **Test:** 25,000 reviews

All three splits are approximately balanced between negative and
positive sentiment.

This section visualizes class distributions and examines review lengths.
Length statistics and examples use training data so that exploratory
analysis does not guide decisions using the held-out test set.

Review lengths are measured using whitespace-separated units in the
original text. This is a simple descriptive measure, not spaCy
tokenization or the subword tokenization used by BERT.

In [ ]:
label_counts = (
    fixed_split_summary
    .set_index("Split")[["Negative", "Positive"]]
)

ax = label_counts.plot(
    kind="bar",
    figsize=(8, 4),
    color=["#d97757", "#4c9f70"],
    rot=0,
)

ax.set_title("Sentiment Label Distribution Across Fixed Splits")
ax.set_xlabel("Split")
ax.set_ylabel("Number of reviews")
ax.legend(title="Sentiment")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "label_distribution.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

In [ ]:
train_description = prepared_frames["train"][
    ["review_id", "text", "label"]
].copy()

train_description["whitespace_word_count"] = (
    train_description["text"]
    .str.split()
    .str.len()
)

length_summary = (
    train_description["whitespace_word_count"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .rename_axis("Statistic")
    .reset_index(name="Whitespace word count")
)

display(length_summary)

length_summary.to_csv(
    OUTPUT_DIR / "training_review_length_summary.csv",
    index=False,
)

In [ ]:
word_counts = train_description["whitespace_word_count"]

# Limit the displayed range to make the main distribution readable.
plot_limit = float(word_counts.quantile(0.99))
visible_counts = word_counts[word_counts <= plot_limit]
omitted_count = int((word_counts > plot_limit).sum())

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(
    visible_counts,
    bins=50,
    color="#547aa5",
    edgecolor="white",
)

ax.axvline(
    word_counts.median(),
    color="#b44b41",
    linestyle="--",
    label=f"Full-data median: {word_counts.median():.0f}",
)

ax.set_title("Training Review Lengths")
ax.set_xlabel("Whitespace-separated word count")
ax.set_ylabel("Number of reviews")
ax.legend()

fig.text(
    0.5,
    0.01,
    (
        f"Display limited to the 99th percentile "
        f"({plot_limit:.0f} words); {omitted_count} longer reviews omitted. "
        "Summary statistics include all training reviews."
    ),
    ha="center",
    fontsize=9,
)

plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.savefig(
    OUTPUT_DIR / "training_review_lengths.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

In [ ]:
sample_parts = []

for label_id in [0, 1]:
    sample_parts.append(
        train_description.loc[
            train_description["label"] == label_id
        ].sample(n=1, random_state=SEED)
    )

training_examples = pd.concat(
    sample_parts,
    ignore_index=True,
)

for _, row in training_examples.iterrows():
    sentiment = "Negative" if row["label"] == 0 else "Positive"

    print(f"Review ID: {row['review_id']}")
    print(f"Sentiment: {sentiment} ({row['label']})")
    print(f"Whitespace word count: {row['whitespace_word_count']}")
    print("Original review excerpt:")
    print(row["text"][:700])

    if len(row["text"]) > 700:
        print("[Excerpt truncated]")

    print()

training_examples.to_csv(
    OUTPUT_DIR / "training_sample_examples.csv",
    index=False,
)

### Dataset Description for the Report

This project uses the English IMDB movie-review dataset distributed
through Hugging Face as `stanfordnlp/imdb`. The original labeled dataset
contains 25,000 training reviews and 25,000 test reviews. Labels indicate
negative sentiment (0) or positive sentiment (1). The additional
50,000 unlabeled reviews are excluded from our supervised experiments.

Quality checks found no missing or empty review texts, missing labels,
unexpected labels, or conflicting labels for identical texts. However,
123 distinct review texts occurred in both original labeled splits.
We removed the corresponding 123 training records, followed by 96
additional duplicate training records.

The remaining 24,781 training-source reviews were divided into 19,824
training reviews and 4,957 validation reviews using a stratified split
with random seed 42. The official 25,000-review test set was preserved.
The final dataset contains 49,781 records.

Negative reviews account for 49.78% of training data, 49.79% of validation
data, and 50.00% of test data. No exact review text or record ID is shared
between the final splits.

The official test set retains 199 extra exact-duplicate rows.
Consequently, some repeated test reviews contribute more than once to
evaluation metrics. Exact-text checks also do not rule out near-duplicate
reviews.

The training-review length table and histogram describe variation in
review length. The displayed examples illustrate positive and negative
reviews.

Dataset source: https://huggingface.co/datasets/stanfordnlp/imdb

Training reviews contain an average of 234.70 whitespace-separated
words, with a median of 175. Lengths range from 10 to 2,470 words.
The middle 50% of reviews contain between 127 and 285 words, while
the 95th percentile is 600 words. The distribution is right-skewed,
with a small number of substantially longer reviews.

For readability, the histogram excludes 199 reviews above the
99th-percentile threshold of 917.77 words. These reviews remain in
the dataset and are included in all summary statistics.

## 7. Minimal Text Cleanup

Some IMDB reviews contain HTML formatting, such as `<br />`, and
HTML character references, such as `&amp;`.

Before linguistic preprocessing, we:

1. Replace HTML tags with spaces.
2. Decode HTML character references into readable characters.
3. Collapse repeated whitespace and remove leading/trailing whitespace.

Replacing tags with spaces prevents adjacent words from being joined.
Decoding character references restores their intended characters.
Whitespace normalization produces consistent spacing.

This cleanup preserves capitalization, punctuation, negation, and
word order. It does not remove stop words or lemmatize words.

We preserve the original `text` field. The minimally cleaned version
will be stored separately as `text_clean` and can later be passed to
a transformer model's own tokenizer.

The cleanup function handles common formatting in this dataset.
It is not a general-purpose HTML parser.

In [ ]:
import html
import re

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
WHITESPACE_PATTERN = re.compile(r"\s+")


def clean_review(text):
    """Remove common HTML formatting and normalize whitespace."""
    # Remove actual tags before decoding entities, so escaped
    # literal text such as &lt;movie&gt; is not treated as a tag.
    without_tags = HTML_TAG_PATTERN.sub(" ", text)
    decoded = html.unescape(without_tags)
    cleaned = WHITESPACE_PATTERN.sub(" ", decoded).strip()

    return cleaned

In [ ]:
# A constructed example illustrates each cleanup operation.
# This is not a review from the dataset.
demo_text = (
    "  I did NOT enjoy this movie!<br /><br />"
    "The acting &amp; dialogue were disappointing.\n\n"
    "  "
)

print("Before cleanup:")
print(repr(demo_text))

print("\nAfter cleanup:")
print(repr(clean_review(demo_text)))

In [ ]:
# Find a training review containing an HTML tag.
has_html_tag = prepared_frames["train"]["text"].str.contains(
    HTML_TAG_PATTERN,
    regex=True,
    na=False,
)

if has_html_tag.any():
    cleanup_example = prepared_frames["train"].loc[
        has_html_tag
    ].iloc[0]

    print(f"Review ID: {cleanup_example['review_id']}")

    print("\nOriginal excerpt:")
    print(repr(cleanup_example["text"][:700]))

    print("\nCleaned excerpt:")
    print(repr(clean_review(cleanup_example["text"])[:700]))
else:
    print("No training review containing an HTML tag was found.")

### Cleanup Interpretation

The constructed example demonstrates that HTML breaks become spaces,
`&amp;` becomes `&`, and repeated whitespace is normalized.

The dataset example illustrates the same function on an actual
training review. Sentiment-bearing words, negation, punctuation,
and capitalization are retained.

At this stage, we have defined and demonstrated cleanup without
modifying the fixed datasets. The function will be applied consistently
to all splits during batch preprocessing.

## 8. Tokenization

Tokenization divides a review into individual units called tokens,
such as words and punctuation.

We use spaCy's English tokenizer rather than splitting only on spaces.
It can separate punctuation and recognize contractions. For example,
a contraction such as "didn't" is separated into "did" and "n't".

Tokenization provides the units needed for subsequent stop-word
removal and lemmatization. Preserving contracted negation is important
because removing it can change sentiment.

We run the complete spaCy linguistic pipeline on the cleaned review
before filtering tokens. This allows the lemmatizer to use the original
sentence context. In this section, we inspect only the tokenization
output; filtering and lemmatization will be explained separately.

This word-level tokenization is intended for the classical preprocessing
pipeline. Transformer models will use their own model-specific tokenizer.

In [ ]:
# Constructed example, not a dataset review.
tokenization_demo = (
    "I didn't enjoy this movie, but the actors were excellent!"
)

demo_doc = nlp(clean_review(tokenization_demo))

demo_token_table = pd.DataFrame({
    "Position": [token.i for token in demo_doc],
    "Token": [token.text for token in demo_doc],
    "Is punctuation": [token.is_punct for token in demo_doc],
    "Is whitespace": [token.is_space for token in demo_doc],
})

display(demo_token_table)

In [ ]:
# Reuse the negative example selected in Section 6.
tokenization_example = training_examples.loc[
    training_examples["label"] == 0
].iloc[0]

example_clean_text = clean_review(tokenization_example["text"])

# Keep this Doc for the next stop-word and lemmatization sections.
example_doc = nlp(example_clean_text)

print(f"Review ID: {tokenization_example['review_id']}")
print(f"Sentiment label: {tokenization_example['label']}")
print(f"Total spaCy tokens: {len(example_doc)}")
print(
    "Whitespace-separated units after cleanup:",
    len(example_clean_text.split()),
)

print("\nFirst 50 tokens:")
print([token.text for token in example_doc[:50]])

### Tokenization Interpretation

The constructed example shows how spaCy separates punctuation and
contracted negation from surrounding words.

The real training example displays the first 50 tokens and compares
spaCy's token count with a whitespace-based count. These measures
differ because they use different definitions of a token.

No tokens have been removed in this section. The complete spaCy Doc
retains linguistic annotations for the subsequent preprocessing steps.

## 9. Stop-word Removal

Stop words are frequent words such as "the", "a", and "is".
Removing them can reduce the number of tokens and the vocabulary size
used by classical text representations such as TF-IDF.

However, some common words carry important sentiment information.
For example, removing "not" from "not good" changes the meaning.

We therefore use spaCy's stop-word flags with explicit exceptions:
- Preserve negation, including "not", "no", "never", and "n't".
- Preserve contrast markers such as "but", "however", and "although".

We also remove punctuation-only and whitespace tokens from the classical
pipeline. This is a separate filtering rule, not stop-word removal.

These choices can discard useful information, such as emphasis expressed
through punctuation. They are not guaranteed to improve classification
performance.

The minimally cleaned text intended for transformer models remains
unchanged.

In [ ]:
# Explicit exceptions to stop-word removal.
PROTECTED_WORDS = {
    "not", "no", "never", "nor", "neither",
    "n't", "n’t", "n‘t",
    "but", "however", "although", "though", "yet",
}


def token_filter_reason(token):
    """Explain why a token is kept or removed."""
    if token.is_space:
        return "remove: whitespace"

    if token.is_punct:
        return "remove: punctuation"

    if token.text.lower() in PROTECTED_WORDS:
        return "keep: protected word"

    if token.is_stop:
        return "remove: stop word"

    return "keep: content token"


def keep_classical_token(token):
    """Select tokens for the classical preprocessing pipeline."""
    return token_filter_reason(token).startswith("keep:")

In [ ]:
demo_filter_table = pd.DataFrame({
    "Token": [token.text for token in demo_doc],
    "spaCy stop word": [token.is_stop for token in demo_doc],
    "Decision": [token_filter_reason(token) for token in demo_doc],
})

display(demo_filter_table)

demo_kept_tokens = [
    token.text
    for token in demo_doc
    if keep_classical_token(token)
]

print("Tokens retained:")
print(demo_kept_tokens)

In [ ]:
# Keep spaCy Token objects so their contextual lemmas remain available.
example_kept_tokens = [
    token
    for token in example_doc
    if keep_classical_token(token)
]

example_filter_table = pd.DataFrame({
    "Token": [token.text for token in example_doc],
    "Decision": [
        token_filter_reason(token)
        for token in example_doc
    ],
})

decision_counts = (
    example_filter_table["Decision"]
    .value_counts()
    .rename_axis("Decision")
    .reset_index(name="Token count")
)

display(decision_counts)

print(f"Original token count: {len(example_doc)}")
print(f"Retained token count: {len(example_kept_tokens)}")
print(
    "Removed token count:",
    len(example_doc) - len(example_kept_tokens),
)

print("\nFirst 50 retained tokens:")
print([token.text for token in example_kept_tokens[:50]])

In [ ]:
# Constructed examples check our sentiment-specific filtering rule.
negation_examples = [
    "The movie was not good.",
    "I never enjoyed it.",
    "There is no interesting story.",
    "I didn't like this movie.",
    "I didn’t like this movie.",
]

negation_check_rows = []

for sentence in negation_examples:
    doc = nlp(clean_review(sentence))

    negation_tokens = [
        token
        for token in doc
        if token.text.lower() in PROTECTED_WORDS
    ]

    assert negation_tokens, (
        f"No protected token detected in: {sentence}"
    )
    assert all(
        keep_classical_token(token)
        for token in negation_tokens
    ), f"Protected token removed in: {sentence}"

    negation_check_rows.append({
        "Sentence": sentence,
        "Retained tokens": [
            token.text
            for token in doc
            if keep_classical_token(token)
        ],
    })

display(pd.DataFrame(negation_check_rows))
print("All negation-protection checks passed.")

### Stop-word Removal Interpretation

The demonstration shows that common function words are removed while
protected negation and contrast words remain.

The real training example reports token counts for each filtering
decision. Punctuation and whitespace removal are recorded separately
from stop-word removal.

The constructed checks confirm that the filtering rules preserve
negation in the tested examples, including straight and curly apostrophe
contractions.

No fixed dataset has been modified yet. The same rules will be applied
to every split during batch preprocessing.

## 10. Lemmatization

Lemmatization maps inflected words to their dictionary base forms,
called lemmas. For example, "actors" can become "actor" and
"enjoyed" can become "enjoy".

This can reduce vocabulary sparsity by allowing related grammatical
forms to share a feature in the classical text representation.
Lemmatization does not merge all related words or synonyms.

We use lemmas calculated by spaCy from the complete cleaned review,
before token filtering. This preserves sentence context for linguistic
analysis.

For the classical pipeline, we also lowercase the resulting lemmas.
Lowercasing is a separate normalization step that merges capitalization
variants.

Contracted negation tokens such as "n't" and "n’t" are normalized to
"not". Other protected words retain their lowercase surface forms,
ensuring that filtering and lemmatization preserve these exceptions.

The original and minimally cleaned reviews remain available.
Lemmatization and lowercasing are applied only to the classical version.

Lemmatization may lose grammatical distinctions and can produce errors.
Its effect on classification accuracy must be evaluated later.

In [ ]:
NEGATION_CONTRACTIONS = {"n't", "n’t", "n‘t"}


def classical_token_form(token):
    """Return the normalized form of a retained spaCy token."""
    surface = token.text.lower()

    # Give straight/curly contracted negation one consistent form.
    if surface in NEGATION_CONTRACTIONS:
        return "not"

    # Preserve our explicit sentiment-related exceptions.
    if surface in PROTECTED_WORDS:
        return surface

    lemma = token.lemma_.strip().lower()

    # Fall back to the surface form if a lemma is empty.
    return lemma if lemma else surface

In [ ]:
demo_lemma_table = pd.DataFrame([
    {
        "Retained token": token.text,
        "Part of speech": token.pos_,
        "spaCy lemma": token.lemma_,
        "Classical form": classical_token_form(token),
    }
    for token in demo_doc
    if keep_classical_token(token)
])

display(demo_lemma_table)

demo_classical_tokens = [
    classical_token_form(token)
    for token in demo_doc
    if keep_classical_token(token)
]

print("Classical tokens:")
print(demo_classical_tokens)

print("\nClassical text:")
print(" ".join(demo_classical_tokens))

In [ ]:
example_lemma_table = pd.DataFrame([
    {
        "Token": token.text,
        "Lowercase surface": token.text.lower(),
        "Part of speech": token.pos_,
        "spaCy lemma": token.lemma_,
        "Classical form": classical_token_form(token),
    }
    for token in example_kept_tokens
])

changed_forms = example_lemma_table.loc[
    example_lemma_table["Lowercase surface"]
    != example_lemma_table["Classical form"]
]

print(
    "Tokens changed beyond lowercasing:",
    len(changed_forms),
)
display(changed_forms.head(25))

example_classical_tokens = [
    classical_token_form(token)
    for token in example_kept_tokens
]
example_classical_text = " ".join(example_classical_tokens)

print("\nFirst 50 classical tokens:")
print(example_classical_tokens[:50])

print("\nClassical text excerpt:")
print(example_classical_text[:700])

In [ ]:
normalized_negation_rows = []

for sentence in negation_examples:
    doc = nlp(clean_review(sentence))

    retained = [
        token
        for token in doc
        if keep_classical_token(token)
    ]
    normalized = [
        classical_token_form(token)
        for token in retained
    ]

    for token in retained:
        surface = token.text.lower()

        if surface in PROTECTED_WORDS:
            expected = (
                "not"
                if surface in NEGATION_CONTRACTIONS
                else surface
            )

            assert classical_token_form(token) == expected, (
                f"Protected word changed unexpectedly: {token.text}"
            )

    assert len(normalized) == len(retained)
    assert all(form.strip() for form in normalized)

    normalized_negation_rows.append({
        "Sentence": sentence,
        "Classical text": " ".join(normalized),
    })

display(pd.DataFrame(normalized_negation_rows))
print("All normalization checks passed.")

### Lemmatization Interpretation

The demonstration illustrates how plural words can be reduced to
singular lemmas. The real review table shows changes produced for
retained tokens using their original linguistic context.

Lowercasing and contracted-negation normalization are recorded as
additional normalization choices, separate from lemmatization.

The checks confirm that protected words remain intact under the
normalization rules and that each retained token produces a non-empty
classical form.

All three required techniques have now been demonstrated:
tokenization, stop-word removal, and lemmatization.

The fixed datasets have not yet been modified. Next, we will apply
the complete pipeline consistently to training, validation, and test
data while preserving original text, labels, and review IDs.

## 11. Apply Preprocessing to All Fixed Splits

We apply the same preprocessing rules to training, validation, and test
reviews without changing their split membership.

Each output record retains:
- The original review ID.
- Original and project split information.
- The original review text.
- The sentiment label.

We add:
- `text_clean`: Minimally cleaned natural text for transformer tokenization.
- `tokens_classical`: Filtered and normalized word-level tokens.
- `text_classical`: Classical tokens joined with spaces.
- Token counts documenting the effect of filtering.

spaCy processes complete cleaned reviews before we select and normalize
tokens. We use `nlp.pipe` to process reviews in batches without storing
every spaCy Doc in memory.

The pretrained linguistic model and fixed rules do not learn from these
reviews. Any later learned representation, such as TF-IDF or Word2Vec,
must be fitted using training data only.

No records are automatically deleted if preprocessing produces empty
text. Such cases will be reported for investigation.

In [ ]:
from tqdm.auto import tqdm
from time import perf_counter

BATCH_SIZE = 64


def classical_features(doc):
    """Extract classical tokens and filtering counts from one Doc."""
    retained_tokens = [
        token
        for token in doc
        if keep_classical_token(token)
    ]

    normalized_tokens = [
        classical_token_form(token)
        for token in retained_tokens
    ]

    return {
        "tokens_classical": normalized_tokens,
        "text_classical": " ".join(normalized_tokens),
        "token_count_original": len(doc),
        "token_count_classical": len(normalized_tokens),
        "stopword_count_removed": sum(
            token_filter_reason(token) == "remove: stop word"
            for token in doc
        ),
        "punctuation_count_removed": sum(
            token.is_punct for token in doc
        ),
        "whitespace_count_removed": sum(
            token.is_space for token in doc
        ),
        "protected_token_count": sum(
            token_filter_reason(token) == "keep: protected word"
            for token in doc
        ),
    }

In [ ]:
processed_frames = {}
processing_times = []

for split_name, source_frame in prepared_frames.items():
    started = perf_counter()

    # Reset the table index while retaining stable review IDs.
    output_frame = source_frame.copy().reset_index(drop=True)

    output_frame["text_clean"] = output_frame["text"].map(
        clean_review
    )

    feature_rows = []

    docs = nlp.pipe(
        output_frame["text_clean"],
        batch_size=BATCH_SIZE,
        n_process=1,
    )

    for doc in tqdm(
        docs,
        total=len(output_frame),
        desc=f"Processing {split_name}",
    ):
        feature_rows.append(classical_features(doc))

    feature_frame = pd.DataFrame(feature_rows)

    # nlp.pipe returns documents in input order.
    assert len(feature_frame) == len(output_frame)

    output_frame = pd.concat(
        [output_frame, feature_frame],
        axis=1,
    )

    processed_frames[split_name] = output_frame

    processing_times.append({
        "Split": split_name,
        "Reviews processed": len(output_frame),
        "Elapsed seconds": round(perf_counter() - started, 2),
    })

display(pd.DataFrame(processing_times))

In [ ]:
preprocessing_rows = []

for split_name, frame in processed_frames.items():
    empty_clean = frame["text_clean"].str.strip().eq("")
    empty_classical = frame["tokens_classical"].map(len).eq(0)

    preprocessing_rows.append({
        "Split": split_name,
        "Reviews": len(frame),
        "Mean original tokens": round(
            frame["token_count_original"].mean(), 2
        ),
        "Mean classical tokens": round(
            frame["token_count_classical"].mean(), 2
        ),
        "Empty cleaned reviews": int(empty_clean.sum()),
        "Empty classical reviews": int(empty_classical.sum()),
    })

preprocessing_summary = pd.DataFrame(preprocessing_rows)
display(preprocessing_summary)

preprocessing_summary.to_csv(
    OUTPUT_DIR / "preprocessing_summary.csv",
    index=False,
)

In [ ]:
for split_name, processed in processed_frames.items():
    source = prepared_frames[split_name].reset_index(drop=True)

    # Original columns, row order, labels, and IDs must be unchanged.
    pd.testing.assert_frame_equal(
        processed[source.columns],
        source,
    )

    # Every input token is either retained or removed by one rule.
    accounted_tokens = (
        processed["token_count_classical"]
        + processed["stopword_count_removed"]
        + processed["punctuation_count_removed"]
        + processed["whitespace_count_removed"]
    )

    assert accounted_tokens.eq(
        processed["token_count_original"]
    ).all(), f"Token accounting failed in {split_name}"

    assert processed["tokens_classical"].map(len).eq(
        processed["token_count_classical"]
    ).all(), f"Classical token count mismatch in {split_name}"

    assert processed["tokens_classical"].map(
        lambda tokens: all(
            isinstance(token, str) and bool(token.strip())
            for token in tokens
        )
    ).all(), f"Invalid classical token in {split_name}"

    assert processed["text_classical"].equals(
        processed["tokens_classical"].map(" ".join)
    ), f"Classical text mismatch in {split_name}"

print("All record-preservation and token-accounting checks passed.")

In [ ]:
# Reuse the same negative training example from earlier sections.
example_id = tokenization_example["review_id"]

processed_example = processed_frames["train"].loc[
    processed_frames["train"]["review_id"] == example_id
].iloc[0]

print(f"Review ID: {processed_example['review_id']}")
print(f"Sentiment label: {processed_example['label']}")

for column, title in [
    ("text", "Original review"),
    ("text_clean", "Minimally cleaned review"),
    ("text_classical", "Classical processed review"),
]:
    print(f"\n{title} — first 700 characters:")
    print(processed_example[column][:700])

print(
    "\nToken counts:",
    processed_example["token_count_original"],
    "→",
    processed_example["token_count_classical"],
)

### Batch Preprocessing Interpretation

The pipeline produces two model-input versions while retaining the
original reviews and labels.

`text_clean` retains natural wording for a transformer model's tokenizer.
`tokens_classical` and `text_classical` contain filtered, lowercased,
lemmatized forms for classical representations.

The summary reports token counts and any empty outputs. Verification
checks confirm that original records and split membership remain
unchanged and that token filtering counts are consistent.

A reduction in token count demonstrates the effect of preprocessing.
It does not establish an improvement in model accuracy.

## 12. Final Verification and Export

Earlier checks confirmed that no identical original review text occurs
across the fixed splits.

However, preprocessing can map different original reviews to the same
text. For example, differences in HTML formatting may disappear during
cleanup, while capitalization and grammatical differences may disappear
during classical preprocessing.

We therefore check cross-split overlap separately for:
- Original text.
- Minimally cleaned text.
- Classical processed text.

We also check whether identical processed texts have conflicting labels.

These checks do not automatically remove records or change split
membership. Any findings will be reviewed before export.

In [ ]:
representation_columns = [
    "text",
    "text_clean",
    "text_classical",
]

representation_overlap_rows = []

for column in representation_columns:
    for left, right in combinations(processed_frames, 2):
        left_frame = processed_frames[left]
        right_frame = processed_frames[right]

        shared_values = (
            set(left_frame[column])
            & set(right_frame[column])
        )

        pair_records = pd.concat(
            [
                left_frame[[column, "label"]],
                right_frame[[column, "label"]],
            ],
            ignore_index=True,
        )

        shared_records = pair_records.loc[
            pair_records[column].isin(shared_values)
        ]

        conflicting_values = (
            shared_records.groupby(column)["label"]
            .nunique()
            .gt(1)
            .sum()
        )

        representation_overlap_rows.append({
            "Text version": column,
            "Split pair": f"{left} / {right}",
            "Distinct shared texts": len(shared_values),
            "Affected rows in left split": int(
                left_frame[column].isin(shared_values).sum()
            ),
            "Affected rows in right split": int(
                right_frame[column].isin(shared_values).sum()
            ),
            "Shared texts with conflicting labels": int(
                conflicting_values
            ),
        })

representation_overlap_summary = pd.DataFrame(
    representation_overlap_rows
)

display(representation_overlap_summary)

representation_overlap_summary.to_csv(
    OUTPUT_DIR / "representation_overlap_summary.csv",
    index=False,
)

In [ ]:
representation_duplicate_rows = []

for split_name, frame in processed_frames.items():
    for column in representation_columns:
        value_counts = frame[column].value_counts()

        conflicting_values = (
            frame.groupby(column)["label"]
            .nunique()
            .gt(1)
            .sum()
        )

        representation_duplicate_rows.append({
            "Split": split_name,
            "Text version": column,
            "Duplicate extra rows": int(
                frame[column].duplicated().sum()
            ),
            "Distinct repeated texts": int(
                value_counts.gt(1).sum()
            ),
            "Texts with conflicting labels": int(
                conflicting_values
            ),
        })

representation_duplicate_summary = pd.DataFrame(
    representation_duplicate_rows
)

display(representation_duplicate_summary)

representation_duplicate_summary.to_csv(
    OUTPUT_DIR / "representation_duplicate_summary.csv",
    index=False,
)

In [ ]:
overlap_record_parts = []

for column in ["text_clean", "text_classical"]:
    for left, right in combinations(processed_frames, 2):
        shared_values = (
            set(processed_frames[left][column])
            & set(processed_frames[right][column])
        )

        for split_name in [left, right]:
            matching_records = processed_frames[split_name].loc[
                processed_frames[split_name][column].isin(shared_values),
                [
                    "review_id",
                    "split",
                    "label",
                    "text",
                    "text_clean",
                    "text_classical",
                ],
            ].copy()

            matching_records["checked_representation"] = column
            matching_records["split_pair"] = f"{left} / {right}"

            overlap_record_parts.append(matching_records)

representation_overlap_records = pd.concat(
    overlap_record_parts,
    ignore_index=True,
)

representation_overlap_records.to_json(
    OUTPUT_DIR / "representation_overlap_records.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print(
    "Overlap inspection entries:",
    len(representation_overlap_records),
)
print(
    "A record may appear more than once if it overlaps "
    "in multiple representations or split pairs."
)

### Final Verification Findings

No original, minimally cleaned, or classical processed text is shared
between training, validation, and test splits.

Training data contains two extra duplicate rows after minimal cleanup
and three after classical preprocessing. Validation data has no repeated
representations. The preserved test set contains 199 extra duplicates
in original text, 201 in cleaned text, and 203 in classical text.

No identical representation within a split has conflicting labels.
We retain these records and keep the fixed split membership unchanged.

### Export Format

Each split is exported as JSON Lines (JSONL), with one review record
per line. This format preserves `tokens_classical` as a list rather
than converting it into a string.

The export includes original and processed text, labels, review IDs,
split information, and preprocessing counts. Metadata records the
configuration and verification findings.

In [ ]:
export_paths = {}

for split_name, frame in processed_frames.items():
    path = OUTPUT_DIR / f"imdb_{split_name}_prepared.jsonl"

    frame.to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    export_paths[split_name] = path

export_summary = pd.DataFrame([
    {
        "Split": split_name,
        "Reviews": len(processed_frames[split_name]),
        "File": path.name,
        "Size (MiB)": round(path.stat().st_size / (1024 ** 2), 2),
    }
    for split_name, path in export_paths.items()
])

display(export_summary)

In [ ]:
import hashlib

export_checks = []
export_hashes = {}

for split_name, path in export_paths.items():
    expected_rows = processed_frames[split_name].to_dict(
        orient="records"
    )

    # Compare every exported record with its in-memory source.
    row_count = 0

    with path.open("r", encoding="utf-8") as file:
        for row_count, line in enumerate(file, start=1):
            assert row_count <= len(expected_rows), (
                f"Unexpected extra record in {split_name}"
            )

            exported_row = json.loads(line)

            assert exported_row == expected_rows[row_count - 1], (
                f"Export mismatch in {split_name}, row {row_count}"
            )

    assert row_count == len(expected_rows), (
        f"Export row count mismatch in {split_name}"
    )

    # Record a checksum for detecting later file changes.
    checksum = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            checksum.update(chunk)

    export_hashes[split_name] = checksum.hexdigest()

    export_checks.append({
        "Split": split_name,
        "Verified records": row_count,
        "Exact record comparison": "Passed",
    })

display(pd.DataFrame(export_checks))
print("All exported records match the prepared data.")

In [ ]:
# Fingerprints identify the loaded dataset objects.
# They are not a substitute for a pinned Hugging Face repository revision.
dataset_fingerprints = {
    split_name: getattr(split_data, "_fingerprint", None)
    for split_name, split_data in dataset.items()
}

metadata = {
    "dataset_name": DATASET_NAME,
    "dataset_source": (
        "https://huggingface.co/datasets/stanfordnlp/imdb"
    ),
    "dataset_revision": None,
    "revision_note": (
        "The initial download did not pin a repository revision. "
        "Reuse these exported files for the current experiments."
    ),
    "source_dataset_fingerprints": dataset_fingerprints,
    "seed": SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "label_mapping": {
        str(key): value
        for key, value in label_mapping.items()
    },
    "package_versions": package_versions,
    "spacy_model": {
        "name": "en_core_web_sm",
        "version": nlp.meta.get("version"),
        "active_components": nlp.pipe_names,
    },
    "split_policy": {
        "preserve_official_test": True,
        "remove_training_overlap_with_test": True,
        "deduplicate_remaining_training_text": True,
        "duplicate_keep_policy": "first",
        "stratify_by": "label",
    },
    "preprocessing": {
        "cleanup": (
            "Replace HTML tags with spaces, decode HTML entities, "
            "and normalize whitespace."
        ),
        "classical": (
            "spaCy tokenization; stop-word removal with exceptions; "
            "punctuation and whitespace filtering; contextual "
            "lemmatization; lowercasing; contracted negation to not."
        ),
        "protected_words": sorted(PROTECTED_WORDS),
        "negation_contractions": sorted(NEGATION_CONTRACTIONS),
        "batch_size": BATCH_SIZE,
    },
    "split_summary": fixed_split_summary.to_dict(orient="records"),
    "removal_summary": removal_summary.to_dict(orient="records"),
    "preprocessing_summary": preprocessing_summary.to_dict(
        orient="records"
    ),
    "representation_overlap": representation_overlap_summary.to_dict(
        orient="records"
    ),
    "representation_duplicates": (
        representation_duplicate_summary.to_dict(orient="records")
    ),
    "exports": {
        split_name: {
            "file": path.name,
            "records": len(processed_frames[split_name]),
            "sha256": export_hashes[split_name],
        }
        for split_name, path in export_paths.items()
    },
}

with (OUTPUT_DIR / "metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# Save useful evidence alongside the prepared data.
supporting_tables = {
    "original_data_quality.csv": quality_summary,
    "original_train_test_overlap.csv": overlap_summary,
    "training_removal_summary.csv": removal_summary,
    "processing_times.csv": pd.DataFrame(processing_times),
}

for filename, table in supporting_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

print("Saved metadata and supporting tables.")

### Using the Prepared Data

Use the provided split files without creating new train/validation/test
splits.

For classical models:
- Use `tokens_classical` or `text_classical`.
- Preserve the prepared tokens when configuring the vectorizer.
- Do not apply an additional stop-word filter.
- Fit TF-IDF vocabulary and IDF weights on training data only.

For transformer models:
- Use `text_clean`.
- Apply the selected model's own tokenizer.
- Do not use the lemmatized classical text.

Use validation data for model selection and test data for final
evaluation. Keep the provided review IDs for prediction tracking and
error analysis.

All records retain their original text and sentiment labels.

In [ ]:
import shutil

# Store the archive outside OUTPUT_DIR so it cannot include itself.
archive_base = OUTPUT_DIR.resolve().parent / "imdb_outputs"

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(OUTPUT_DIR.resolve()),
)

print(f"Archive created: {archive_path}")

# Download from Google Colab.
from google.colab import files
files.download(archive_path)

## 13. Report Draft

The following sections summarize the project motivation, dataset,
and completed data preparation.

Model experiments are planned but have not been evaluated in this
notebook. The Conclusion is therefore provisional and must be updated
with the group's experimental findings before final submission.

### 13.1 Introduction

Sentiment analysis is an NLP task that identifies opinions or attitudes
expressed in text. In movie reviews, it can be formulated as binary
classification: predicting whether a review expresses positive or
negative sentiment. Understanding review sentiment can support the
analysis of audience feedback.

This project investigates sentiment classification using English
movie reviews from the IMDB dataset. Although reviews often contain
explicit opinion words, sentiment can also depend on negation, contrast,
and context. A review may express both praise and criticism, making
classification more complex than identifying individual positive or
negative words.

The project plans to compare TF-IDF representations with a
transformer-based approach. The proposed experiments include
Logistic Regression or Support Vector Machines using TF-IDF and a
LoRA-fine-tuned DistilBERT or BERT-base model. The final model choices
and performance comparisons will be documented after experimentation.

This notebook focuses on data preparation. It establishes fixed
training, validation, and test sets; examines dataset characteristics;
and implements tokenization, stop-word removal, and lemmatization.
It preserves both minimally cleaned reviews for transformer models
and processed word-level text for classical models.

### 13.2 Dataset Description

#### Source and task

We use `stanfordnlp/imdb`, the Hugging Face distribution of the
Large Movie Review Dataset introduced by Maas et al. (2011).
The task is binary sentiment classification of English movie reviews.

Each labeled record contains:
- `text`: The original review.
- `label`: Negative sentiment (0) or positive sentiment (1).

The original dataset provides 25,000 labeled training reviews,
25,000 labeled test reviews, and 50,000 additional unlabeled reviews.
The unlabeled split is excluded from our supervised experiments.

#### Quality checks and duplicate policy

The original labeled splits contained no missing, non-string, or empty
review texts, missing labels, or unexpected labels. Exact-text checks
also found no identical reviews with conflicting labels.

However, the original training split contained 96 extra duplicate rows,
and the test split contained 199. Additionally, 123 distinct review texts
appeared in both training and test data.

We preserved the official test set and removed the 123 overlapping
training records. We then removed 96 remaining duplicate training
records, retaining the first occurrence of each text. This left
24,781 reviews eligible for training and validation.

#### Fixed splits and label distribution

The eligible reviews were divided into training and validation sets
using an 80%/20% stratified split with random seed 42.

| Split | Reviews | Negative | Positive | Negative (%) | Positive (%) |
|---|---:|---:|---:|---:|---:|
| Training | 19,824 | 9,869 | 9,955 | 49.78 | 50.22 |
| Validation | 4,957 | 2,468 | 2,489 | 49.79 | 50.21 |
| Test | 25,000 | 12,500 | 12,500 | 50.00 | 50.00 |
| Total | 49,781 | 24,837 | 24,944 | — | — |

The classes are approximately balanced. Review IDs and split membership
were saved to a manifest so that subsequent experiments can use the
same records.

#### Review lengths and examples

Training reviews contain an average of 234.70 whitespace-separated
words and a median of 175. Lengths range from 10 to 2,470 words.
The middle 50% contain between 127 and 285 words, and the 95th
percentile is 600 words.

The length distribution is right-skewed. For readability, the histogram
excludes 199 reviews above the 99th-percentile threshold of 917.77 words.
These records remain in the dataset and in the summary statistics.
Whitespace word counts are descriptive measures, not BERT subword counts.

The following training excerpts illustrate the two sentiment classes:

| Review ID | Label | Original review excerpt |
|---|---|---|
| train_09218 | Negative (0) | “This Hal Roach comedy short, A Tough Winter, is the ninety-ninth in the ‘Our Gang/Little Rascals’ series and the eleventh talkie.” |
| train_17779 | Positive (1) | “This is a very good Spanish movie but I am worried that many out there will feel lost” |

The labels apply to the complete reviews. Short excerpts may not contain
all the evidence supporting a review's sentiment.

#### Preprocessing and output versions

Minimal cleanup replaces HTML tags with spaces, decodes HTML character
references, and normalizes whitespace. It preserves capitalization,
punctuation, negation, and word order.

The classical preprocessing pipeline implements three main techniques:

1. **Tokenization:** spaCy separates text into word and punctuation
   tokens, including contracted negation.
2. **Stop-word removal:** Common function words are removed, with
   explicit exceptions for negation and contrast markers.
   Punctuation-only and whitespace tokens are filtered separately.
3. **Lemmatization:** Contextual lemmas reduce grammatical variants
   to base forms. Lemmas are lowercased, and contracted negation
   is normalized to `not`.

spaCy analyzes complete cleaned reviews before token filtering.
The same fixed rules are applied to every split.

| Split | Mean tokens before filtering | Mean classical tokens |
|---|---:|---:|
| Training | 272.71 | 111.57 |
| Validation | 268.19 | 109.58 |
| Test | 265.39 | 108.45 |

The initial counts include spaCy tokens from cleaned reviews, including
punctuation. No review became empty after cleanup or classical
preprocessing.

Each exported record retains original text and includes minimally
cleaned text, classical tokens, classical text, sentiment labels,
review IDs, split information, and preprocessing counts.

#### Final verification and limitations

No original, cleaned, or classical text is shared between the fixed
splits. Preprocessing introduced two extra repeated cleaned-text rows
and three extra repeated classical-text rows within training.
These records were retained because their original texts differ.

The preserved test set contains 199 extra repeated original-text rows,
201 cleaned-text rows, and 203 classical-text rows. No repeated
representation within any split has conflicting labels.

Exact matching does not rule out near-duplicate reviews. Repeated test
records contribute more than once to evaluation metrics. Preprocessing
can also discard information expressed through punctuation or grammar.

The three JSONL exports were read back and compared record by record
with the prepared data. All comparisons passed. Metadata records
package versions, preprocessing rules, split counts, and file checksums.
The initial dataset download did not pin a repository revision;
the saved exports therefore define the data used for this project's
current experiments.

### 13.3 Conclusion — Data Preparation Stage

This work prepared an IMDB dataset for binary sentiment classification.
After removing training records that overlapped with the official test
set and deduplicating the remaining training data, 49,781 labeled
reviews were retained. Fixed training, validation, and test sets were
created with approximately balanced sentiment distributions.

Tokenization, stop-word removal, and lemmatization were implemented
and explained. Negation and selected contrast markers were protected.
Separate text versions support classical and transformer-based models
while retaining the original reviews.

Verification confirmed that labels, review IDs, and split membership
were preserved, no processed reviews were empty, and no identical text
representation crossed split boundaries. The exported files matched
the prepared records exactly.

These results establish a verified foundation for model training.
They do not demonstrate which representation or model performs best.
The final project conclusion will be updated with experimental results,
error-analysis findings, and the effects of the preprocessing choices.

### References

Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y.,
& Potts, C. (2011). Learning Word Vectors for Sentiment Analysis.
Proceedings of the 49th Annual Meeting of the Association for
Computational Linguistics: Human Language Technologies, 142–150.
https://aclanthology.org/P11-1015/

Stanford NLP. IMDB dataset, Hugging Face Datasets.
https://huggingface.co/datasets/stanfordnlp/imdb